In [2]:
pip install git+https://github.com/mimoralea/gym-walk#egg=gym-walk

  Cloning https://github.com/mimoralea/gym-walk to /tmp/pip-install-72c66m_w/gym-walk_9d5658761a6b4e82bfc425066cc9c24b
  Running command git clone --filter=blob:none --quiet https://github.com/mimoralea/gym-walk /tmp/pip-install-72c66m_w/gym-walk_9d5658761a6b4e82bfc425066cc9c24b
  Resolved https://github.com/mimoralea/gym-walk to commit c453a5da6a37b6fadb393c490261bd6b9fa5bfc5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.4/624.4 kB 11.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gym-walk: filename=gym_walk-0.0.3-py3-none-any.whl size=5289 sha256=cfbe40f113c338b553a7a620b5bb4c16bf7e03e1c55bac19739b243d3dcdcc8f
  Stored in directory: /tmp/pip-ephem-wheel-cache-6ug7bz7g/wheels/bf/23/e5/a94be4a90dd18f7ce958c21f192276cb01ef0daaf

In [3]:
import warnings ; warnings.filterwarnings('ignore')

import gym
import numpy as np

import random
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123);

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:


def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi(s)
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")


In [5]:
def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [6]:
def probability_success(env, pi, goal_state, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        while not done and steps < max_steps:
            state, _, done, h = env.step(pi(state))
            steps += 1
        results.append(state == goal_state)
    return np.sum(results)/len(results)

In [7]:
def mean_return(env, pi, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        results.append(0.0)
        while not done and steps < max_steps:
            state, reward, done, _ = env.step(pi(state))
            results[-1] += reward
            steps += 1
    return np.mean(results)

**Frozen Lake MDP**

In [8]:
env = gym.make('FrozenLake-v1')
P = env.env.P
init_state = env.reset()
goal_state = 15
LEFT, DOWN, RIGHT, UP = range(4)

In [9]:
P

{0: {0: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  2: [(0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)],
  3: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False)],
  2: [(0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  3: [(0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 2:

In [10]:
init_state

0

In [11]:
state, reward, done, info = env.step(RIGHT)
print("state:{0} - reward:{1} - done:{2} - info:{3}".format(state, reward, done, info))

state:4 - reward:0.0 - done:False - info:{'prob': 0.3333333333333333}


In [14]:
pi_frozenlake = lambda s: {
    0: RIGHT,
    1: DOWN,
    2: DOWN,
    3: LEFT,
    4: UP,
    5: LEFT,
    6: DOWN,
    7: LEFT,
    8: RIGHT,
    9: UP,
    10: DOWN,
    11: DOWN,
    12: RIGHT,
    13: RIGHT,
    14: DOWN,
    15: LEFT
}[s]
print_policy(pi_frozenlake, P, action_symbols=('<', 'v', '>', '^'), n_cols=4)

Policy:
| 00      > | 01      v | 02      v | 03      < |
| 04      ^ |           | 06      v |           |
| 08      > | 09      ^ | 10      v |           |
|           | 13      > | 14      v |           |


In [15]:
print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, pi_frozenlake, goal_state=goal_state) * 100,
    mean_return(env, pi_frozenlake)))

Reaches goal 1.00%. Obtains an average undiscounted return of 0.0100.


In [16]:
policy=lambda s: {
    0: RIGHT,
    1: DOWN,
    2: RIGHT,
    3: LEFT,
    4: DOWN,
    5: LEFT,
    6: RIGHT,
    7:LEFT,
    8: UP,
    9: DOWN,
    10:LEFT,
    11:DOWN,
    12:RIGHT,
    13:RIGHT,
    14:DOWN,
    15:LEFT
}[s]

print("Name: Easwari M")
print("Register Number: 212223240033")
print_policy(policy, P, action_symbols=('<', 'v', '>', '^'), n_cols=4)

Name: Easwari M
Register Number: 212223240033
Policy:
| 00      > | 01      v | 02      > | 03      < |
| 04      v |           | 06      > |           |
| 08      ^ | 09      v | 10      < |           |
|           | 13      > | 14      v |           |


In [17]:
# Find the probability of success and the mean return of you your policy
print('Your policy reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, policy, goal_state=goal_state) * 100,
    mean_return(env, policy)))

# Compare your policy with the first policy

Your policy reaches goal 10.00%. Obtains an average undiscounted return of 0.1000.



**Policy Evaluation**

In [18]:
def policy_evaluation(pi, P, gamma=1.0, theta=1e-10):
    V = np.zeros(len(P), dtype=np.float64)
    while True:
        prev_V = np.copy(V)
        for s in range(len(P)):
            v = 0
            a = pi(s)
            for prob, next_state, reward, done in P[s][a]:
                v += prob * (reward + gamma * prev_V[next_state] * (1 - done))
            V[s] = v
        if np.max(np.abs(prev_V - V)) < theta:
            break
    return V

# Code to evaluate the first policy
V1 = policy_evaluation(pi_frozenlake, P,gamma=0.99)
print_state_value_function(V1, P, n_cols=4, prec=5, title='State-value function for pi_frozenlake:')

# Code to evaluate the second policy
V2 = policy_evaluation(policy, P)
print_state_value_function(V2, P, n_cols=4, prec=5, title='State-value function for custom policy:')

State-value function for pi_frozenlake:
| 00 0.01102 | 01 0.01695 | 02 0.04033 | 03 0.01987 |
| 04 0.00543 |           | 06 0.08541 |           |
| 08 0.03364 | 09 0.09651 | 10 0.25881 |           |
|           | 13 0.38629 | 14 0.68777 |           |
State-value function for custom policy:
| 00 0.13333 | 01 0.09462 | 02 0.15054 | 03 0.07527 |
| 04 0.17204 |           | 06 0.22581 |           |
| 08 0.34409 | 09 0.51613 | 10 0.52688 |           |
|           | 13 0.67742 | 14 0.83871 |           |


In [26]:
V1>=V2


if(np.sum(V1)>np.sum(V2)):
  print("The first policy is the better policy")
elif(np.sum(V2)>np.sum(V1)):
  print("The second policy is the better policy")
else:
  print("Both policies have their merits.")

The second policy is the better policy
